# 03 - Feature Engineering

El objetivo de este notebook es crear las métricas de negocio que posteriormente serán utilizadas en SQL y Power BI para el análisis de inventario y procurement.

In [1]:
#Importar librerias
import pandas as pd
import numpy as np

In [2]:
#Cargar dataset
df = pd.read_csv("../Datasets/supply_chain_dataset1.csv")

In [3]:
#Convertir nombres de columnas a minúsculas

df.columns = df.columns.str.lower()

df.columns

Index(['date', 'sku_id', 'warehouse_id', 'supplier_id', 'region', 'units_sold',
       'inventory_level', 'supplier_lead_time_days', 'reorder_point',
       'order_quantity', 'unit_cost', 'unit_price', 'promotion_flag',
       'stockout_flag', 'demand_forecast'],
      dtype='str')

In [6]:
#Transformar fecha en date format y boolean promotion_flag y stockout_flag

df["date"] = pd.to_datetime(df["date"])
df["promotion_flag"] = df["promotion_flag"].astype(bool)
df["stockout_flag"] = df["stockout_flag"].astype(bool)

In [7]:
df.dtypes

date                       datetime64[us]
sku_id                                str
warehouse_id                          str
supplier_id                           str
region                                str
units_sold                          int64
inventory_level                     int64
supplier_lead_time_days             int64
reorder_point                       int64
order_quantity                      int64
unit_cost                         float64
unit_price                        float64
promotion_flag                       bool
stockout_flag                        bool
demand_forecast                   float64
dtype: object

### Revisión de tipos de datos

Se ha verificado que las variables numéricas, categóricas y temporales presentan el formato adecuado para el análisis.

Además, las variables binarias (promotion_flag y stockout_flag) se convierten a tipo booleano para mejorar la legibilidad y consistencia del modelo de datos.

## Evaluación de Variables
Stockout_flag no presenta variabilidad, todos los registros contiene valor cero. 

### KPI 1: Inventory Value 
¿Cuánto capital tenemos inmovilizado en inventario?

In [8]:
df["inventory_value"] = (df["inventory_level"] * df["unit_cost"])

In [9]:
df[["inventory_level", "unit_cost", "inventory_value"]].head()

,inventory_level,unit_cost,inventory_value
0,592,13.95,8258.40
1,575,13.95,8021.25
2,540,13.95,7533.00
3,516,13.95,7198.20
4,495,13.95,6905.25


In [11]:
df["inventory_value"].describe().round(2)

count    91250.00
mean      5756.10
std       2788.54
min       1094.16
25%       3521.07
50%       5247.55
75%       7526.79
max      19186.96
Name: inventory_value, dtype: float64

### Conclusiones

Esta métrica representa el valor monetario del inventario disponible.

Se calcula multiplicando el nivel de inventario por el coste unitario de cada producto.

Esta métrica permitirá identificar qué SKUs, almacenes o proveedores concentran una mayor cantidad de capital inmovilizado.

El valor medio del inventario es de aproximadamente 5.756 unidades monetarias por registro.

La diferencia entre el valor mínimo (1.094) y el máximo (19.187) indica que existen productos o situaciones que requieren una inversión significativamente mayor en inventario.

### KPI 2: Procurement Spend

¿Cuánto dinero estamos gastando en aprovisionamiento?

In [13]:
df["procurement_spend"] = (df["order_quantity"] * df["unit_cost"])

In [14]:
df[["order_quantity", "unit_cost", "procurement_spend"]].head()

,order_quantity,unit_cost,procurement_spend
0,0,13.95,0.0
1,0,13.95,0.0
2,0,13.95,0.0
3,0,13.95,0.0
4,0,13.95,0.0


In [16]:
df["procurement_spend"].describe().round(2)

count    91250.00
mean       235.26
std       1076.88
min          0.00
25%          0.00
50%          0.00
75%          0.00
max       9805.62
Name: procurement_spend, dtype: float64

In [17]:
df[df["procurement_spend"] > 0]["procurement_spend"].describe().round(2)

count    5027.00
mean     4270.45
std      1954.14
min      1040.30
25%      2675.10
50%      3966.56
75%      5582.05
max      9805.62
Name: procurement_spend, dtype: float64

### Conclusiones

Esta métrica representa el gasto asociado a los pedidos realizados a proveedores.

Se calcula multiplicando la cantidad solicitada por el coste unitario del producto.

Este KPI permitirá identificar qué proveedores concentran un mayor volumen de gasto y analizar posibles dependencias dentro de la cadena de suministro.

La mayor parte de los registros no presentan actividad de compra, ya que las reposiciones de inventario representan una proporción reducida del total de observaciones.

Sin embargo, cuando se producen pedidos de aprovisionamiento, el gasto puede alcanzar valores superiores a 9.800 unidades monetarias, lo que evidencia la importancia de monitorizar estas operaciones y su impacto financiero.

Además, hemos filtrado únicamente los registros con actividad de compra y se observa que existen 5.027 eventos de aprovisionamiento.

El gasto medio por pedido es de aproximadamente 4.270 unidades monetarias, mientras que el pedido más elevado alcanza los 9.806.

Estos resultados indican que las compras representan eventos puntuales pero de alto impacto financiero, por lo que resulta relevante monitorizar su comportamiento y distribución entre proveedores.

### KPI 3: Forecast Error

¿La previsión de demanda es fiable?

In [18]:
df["forecast_error"] = (df["units_sold"] - df["demand_forecast"])

In [19]:
df["forecast_error"].describe().round(2)

count    91250.00
mean        -0.03
std          2.99
min        -14.08
25%         -2.03
50%         -0.01
75%          1.99
max         13.00
Name: forecast_error, dtype: float64

In [20]:
# Vamos a calcular ahora el forecast error en valor absoluto, ya que permite evaluar la precisión real del forecast y cuantificar cuánto se desvía la demanda prevista respecto a la demanda real.
df["abs_forecast_error"] = df["forecast_error"].abs()
df["abs_forecast_error"].describe().round(2)

count    91250.00
mean         2.38
std          1.81
min          0.00
25%          0.95
50%          2.00
75%          3.44
max         14.08
Name: abs_forecast_error, dtype: float64

### Conclusiones

El análisis muestra que las previsiones de demanda presentan un nivel de precisión elevado. El error medio absoluto es de 2,38 unidades, mientras que las ventas medias se sitúan en torno a 20 unidades por registro.

Esto indica que las desviaciones entre la demanda prevista y la demanda real son relativamente reducidas, por lo que el forecast puede considerarse una herramienta fiable para apoyar las decisiones de inventario y aprovisionamiento.

### KPI 4: Inventory Coverage

Si mañana dejáramos de comprar, ¿cuántos días podría cubrir el inventario actual con la demanda prevista?

In [21]:
df["inventory_coverage"] = (df["inventory_level"] / df["demand_forecast"])

In [22]:
df[["inventory_level", "demand_forecast", "inventory_coverage"]].head()

,inventory_level,demand_forecast,inventory_coverage
0,592,8.52,69.483568
1,575,18.63,30.864198
2,540,39.62,13.629480
3,516,19.43,26.556871
4,495,18.70,26.470588


In [ ]:
df["inventory_coverage"].describe().round(2)

c:\Users\pilir\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    91250.00
mean          inf
std           NaN
min          3.72
25%         16.18
50%         23.65
75%         37.67
max           inf
Name: inventory_coverage, dtype: float64

In [24]:
#Vamos a calcular las medidas estadísticas, considerando demand_forecast mayor que cero; ya que en el análisis anterior no aparece inf en el maximo.
df["inventory_coverage"] = np.where(
    df["demand_forecast"] > 0,
    df["inventory_level"] / df["demand_forecast"],
    np.nan
)

In [25]:
df["inventory_coverage"].describe().round(2)

count    90358.00
mean        45.13
std        477.18
min          3.72
25%         16.12
50%         23.49
75%         37.02
max      74900.00
Name: inventory_coverage, dtype: float64

In [ ]:
#Vamos a comprobar esos valores extremos para posteriormente decidir que hacer: si mantenerlos o filtrarlos.  
df["inventory_coverage"].nlargest(10)

7531     74900.0
11559    64100.0
18821    44200.0
5421     40600.0
81291    30600.0
8273     30100.0
18168    28600.0
2467     26150.0
13484    26000.0
75782    20100.0
Name: inventory_coverage, dtype: float64

- Intuimos que los valores tan altos en inventory coverage vienen de enventory_level alto y demand_forecast bajo; lo cual no es real. 
Por lo que, vamos a analizar si este problema viene de forecast extremadamente bajos. 


In [27]:
df.loc[
    df["inventory_coverage"] > 1000,
    [
        "sku_id",
        "inventory_level",
        "demand_forecast",
        "inventory_coverage"
    ]
].head(10)

,sku_id,inventory_level,demand_forecast,inventory_coverage
663,SKU_1,678,0.16,4237.500000
987,SKU_1,523,0.41,1275.609756
2151,SKU_2,241,0.23,1047.826087
2467,SKU_2,523,0.02,26150.000000
2826,SKU_2,447,0.24,1862.500000
2845,SKU_2,612,0.41,1492.682927
3232,SKU_2,539,0.53,1016.981132
3535,SKU_2,500,0.32,1562.500000
3573,SKU_2,630,0.58,1086.206897
4255,SKU_3,698,0.28,2492.857143


### Conclusiones

Esta métrica indica cuántos días de demanda podrían cubrirse con el inventario disponible.

Valores elevados pueden indicar exceso de stock, mientras que valores reducidos pueden señalar la necesidad de realizar reposiciones para evitar problemas de disponibilidad.

Durante el cálculo de la cobertura de inventario se identificaron registros con demanda prevista igual a cero.

Para evitar divisiones por cero y valores infinitos, estos casos se sustituyen por valores nulos (NaN), ya que la cobertura no puede interpretarse cuando no existe demanda prevista.

La cobertura media de inventario es de 45 días, aunque la mediana se sitúa en 23 días.

La diferencia entre ambas métricas sugiere la existencia de algunos registros con coberturas excepcionalmente elevadas, probablemente asociadas a productos con una demanda prevista muy baja.

Por este motivo, la mediana representa una medida más fiable del comportamiento típico de la cobertura de inventario.

Se identificaron valores muy elevados de cobertura de inventario. Tras la revisión, se comprobó que estos casos están asociados a productos con una demanda prevista extremadamente baja.

La fórmula de cálculo es correcta; sin embargo, cuando la demanda prevista se aproxima a cero, la cobertura puede alcanzar valores muy elevados que no representan una situación operativa habitual.

Por este motivo, para el análisis de negocio se utilizará principalmente la mediana de la cobertura de inventario, ya que representa de forma más fiable el comportamiento típico del inventario.

### KPI 5: Replenishment Flag

¿Con qué frecuencia se realizan pedidos de reposición de inventario?

Esta métrica permite identificar los momentos en los que se ha generado una orden de compra para reponer inventario.

Será útil para analizar:

- Frecuencia de aprovisionamiento.
- Comportamiento de las compras.
- Patrones de reposición por proveedor.
- Relación entre inventario y pedidos.

In [28]:
df["replenishment_flag"] = np.where(
    df["order_quantity"] > 0,
    1,
    0
)

In [29]:
df["replenishment_flag"].value_counts()

replenishment_flag
0    86223
1     5027
Name: count, dtype: int64

### Conclusiones

Un valor de 1 indica que se ha realizado un pedido de compra, mientras que un valor de 0 indica ausencia de actividad de aprovisionamiento.

Este indicador permitirá analizar la frecuencia de reposición y el comportamiento de las compras dentro de la cadena de suministro.

Se identificaron 5.027 eventos de reposición de inventario, lo que representa aproximadamente el 5,5% del total de registros analizados.

Esto indica que la actividad de aprovisionamiento se concentra en momentos específicos y no se produce de forma continua, un comportamiento coherente con una estrategia de reabastecimiento basada en niveles de inventario y puntos de pedido previamente definidos.

### Resumen de KPIs creados

Durante esta fase se han desarrollado nuevas métricas orientadas al análisis de inventario y aprovisionamiento:

- Inventory Value
- Procurement Spend
- Forecast Error
- Absolute Forecast Error
- Inventory Coverage
- Replenishment Flag

Estas variables servirán como base para el análisis posterior en SQL y para el desarrollo del dashboard en Power BI.

### Exportación del dataset limpio

Una vez finalizado el proceso de Feature Engineering, se exporta el dataset enriquecido para su utilización en las siguientes fases del proyecto, incluyendo la integración con APIs externas, análisis en SQL y desarrollo del dashboard en Power BI.

In [30]:
df.to_csv("../Datasets/SC_clean_dataset.csv", index=False)